In [23]:
# wandbのライブラリをimport
import wandb

# wandbへログイン
wandb.login()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/qiminghe/.netrc
wandb: Currently logged in as: qhe202509 (qhe202509-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [24]:
# Pandasのライブラリをインポート
import pandas as pd

# 学習データを読み込んで変数 train に格納
train = pd.read_csv('./train.csv')

# 学習データの表示
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [25]:
# テストデータを読み込んで変数 test に格納
test = pd.read_csv('./test.csv')

# テストデータの表示
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [26]:
# 学習データとテストデータ数を確認
print(train.shape)
print(test.shape)

(891, 12)
(418, 11)


In [27]:
# trainの欠損値の数を調査して、表示する
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [28]:
# testの欠損値の数を調査して、表示する
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [29]:
# 新しい空のDataFrameを作成する
temp = pd.DataFrame()

# 学習データとテストデータのAge列を連結させ、tempにAge列として追加する
temp['Age'] = pd.concat([train['Age'], test['Age']])

# trainとtestのAge列について、欠損値をtempのAge列の平均値で埋める
train['Age'] = train['Age'].fillna(temp['Age'].mean())
test['Age'] = test['Age'].fillna(temp['Age'].mean())

In [30]:
# 学習データとテストデータのFare列を連結させ、tempにFare列として追加する
temp['Fare'] = pd.concat([train['Fare'], test['Fare']])

# testのみに存在するFare列の欠損値をtempのFare列の平均値で埋める
test['Fare'] = test['Fare'].fillna(temp['Fare'].mean())

In [31]:
# 学習データとテストデータのEmbarked列を連結させ、tempにEmbarked列として追加する
temp['Embarked'] = pd.concat([train['Embarked'], test['Embarked']])

# tempのEmbakedの値を集計する
temp['Embarked'].value_counts()

Embarked
S    914
C    270
Q    123
Name: count, dtype: int64

In [32]:
# trainのみに存在するEmbarked列の欠損値を'S'で埋める
train['Embarked'] = train['Embarked'].fillna('S')

In [33]:
# trainとtestから、Cabin,Name,Ticket列を削除する
train = train.drop(columns=['Cabin', 'Name', 'Ticket'])
test = test.drop(columns=['Cabin', 'Name', 'Ticket'])

In [34]:
# trainのSex列とEmbarked列をダミー変数化して、変数train2に格納する
train2 = pd.get_dummies(data=train, columns=['Sex', 'Embarked'])
print(train2.head())
# testのSex列とEmbarked列をダミー変数化して、変数test2に格納する
test2 = pd.get_dummies(data=test, columns=['Sex', 'Embarked'])
print(test2.head())

   PassengerId  Survived  Pclass   Age  SibSp  Parch     Fare  Sex_female  \
0            1         0       3  22.0      1      0   7.2500       False   
1            2         1       1  38.0      1      0  71.2833        True   
2            3         1       3  26.0      0      0   7.9250        True   
3            4         1       1  35.0      1      0  53.1000        True   
4            5         0       3  35.0      0      0   8.0500       False   

   Sex_male  Embarked_C  Embarked_Q  Embarked_S  
0      True       False       False        True  
1     False        True       False       False  
2     False       False       False        True  
3     False       False       False        True  
4      True       False       False        True  
   PassengerId  Pclass   Age  SibSp  Parch     Fare  Sex_female  Sex_male  \
0          892       3  34.5      0      0   7.8292       False      True   
1          893       3  47.0      1      0   7.0000        True     False   
2     

In [35]:
# train2の欠損値の数を調査して、表示する
train2.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Age            0
SibSp          0
Parch          0
Fare           0
Sex_female     0
Sex_male       0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
dtype: int64

In [36]:
# test2の欠損値の数を調査して、表示する
test2.isnull().sum()

PassengerId    0
Pclass         0
Age            0
SibSp          0
Parch          0
Fare           0
Sex_female     0
Sex_male       0
Embarked_C     0
Embarked_Q     0
Embarked_S     0
dtype: int64

In [37]:
# train2の各列の型を表示する
train2.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Sex_female        bool
Sex_male          bool
Embarked_C        bool
Embarked_Q        bool
Embarked_S        bool
dtype: object

In [38]:
# numpyのimport
import numpy as np

# train2をX_trainとY_trainに分ける
X_train = np.array(train2.drop(columns=['Survived'])).astype('float32')
Y_train = np.array(train2['Survived']).astype('float32')

# test2のデータ全体をX_testに格納する
X_test = np.array(test2).astype('float32')

In [39]:
# X_trainとY_trainの3割をX_validとY_validに分割する
from sklearn.model_selection import train_test_split

X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.3, random_state=0)

In [40]:
# 学習データと検証データ、テストデータの形状を確認
print("X_train=", X_train.shape, ", Y_train=", Y_train.shape)
print("X_valid=", X_valid.shape, ", Y_valid=", Y_valid.shape)
print("X_test=", X_test.shape)

X_train= (623, 11) , Y_train= (623,)
X_valid= (268, 11) , Y_valid= (268,)
X_test= (418, 11)


In [41]:
# tensorflowのimport
import tensorflow as tf

In [42]:
# モデルの構築と学習を定義する関数
def train_model():
    # wandbの初期設定
    wandb.init(
        # wandbでのプロジェクト名
        project="kaggle-titanic",
        # wandbで記録してもらいたい設定値
        config={
            "input_dense_shape": 8,
            "hidden_dense_shape": 8,
            "optimizer": "rmsprop",
            "batch_size": 32
        })

    # モデルの初期化とレイヤー定義
    model = tf.keras.Sequential([
        # 入力層 (Inputオブジェクトを使用)
        tf.keras.Input(shape=(11,)),
        tf.keras.layers.Dense(wandb.config.input_dense_shape, activation='relu'),
        # 隠れ層
        tf.keras.layers.Dense(wandb.config.hidden_dense_shape, activation='relu'),
        # 出力層
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    # モデルの構築
    model.compile(optimizer=wandb.config.optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # 学習の実施
    log = model.fit(X_train, Y_train, epochs=5000, batch_size=wandb.config.batch_size, verbose=True,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                                min_delta=0, patience=100,
                                                                verbose=1),
                              wandb.keras.WandbMetricsLogger(log_freq='epoch')
                              ],
                    validation_data=(X_valid, Y_valid))

In [43]:
# train_modelを実行する
train_model()

Epoch 1/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3900 - loss: 115.8914 - val_accuracy: 0.3694 - val_loss: 90.3962
Epoch 2/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3884 - loss: 74.7434 - val_accuracy: 0.3769 - val_loss: 55.0814
Epoch 3/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3852 - loss: 40.0225 - val_accuracy: 0.4067 - val_loss: 22.0803
Epoch 4/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4687 - loss: 8.9061 - val_accuracy: 0.5933 - val_loss: 1.1934
Epoch 5/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6019 - loss: 1.3311 - val_accuracy: 0.5112 - val_loss: 1.9259
Epoch 6/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6180 - loss: 1.2770 - val_accuracy: 0.5672 - val_loss: 1.1584
Epoch 7/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6100 - loss: 1.1813 - val_accuracy: 0.7239 - val_loss: 0.8723
Epoch 8/5000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6180 - loss: 1.1845 - val_accur

In [44]:
# wandbの動作を終了させる
wandb.finish()

epoch/accuracy,▁▁▁▄▅▆▅▆▆▇▇▇▆▆▇▇▆▇▇██▇▆█▆▇▇▇▇█▇▇▇▇██▇▇▇▇
epoch/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▂▆▆▅▇▁▆█▅▆▆▆▇██▇▇█▆▇▃▇▆▆▇▇▄▅█▃█▇██▆▆▅▇▇█
epoch/val_loss,▃█▂▂▅▁▂▃▁▂▂▁▃▁▁▁▃▅▃▁▂▃▁▃▁▂▃▁▃▁▂▁▂▁▁▁▂▁▂▁
epoch/accuracy,0.79775
epoch/epoch,649
epoch/learning_rate,0.001
epoch/loss,0.48347
epoch/val_accuracy,0.78731


In [45]:
# wandbでsweep（グリッドサーチ）を行なうための設定
sweep_config = {
    'method': 'grid',
    'name': 'kaggle-titanic-sweep',
    'metric': {
        'goal': 'maximize',
        'name': 'accuracy'
    },
    'parameters': {
        'input_dense_shape': {'values': [8, 16, 24]},
        'hidden_dense_shape': {'values': [8, 16, 24]},
        'optimizer': {'values': ['sgd', 'rmsprop', 'adam']},
        'batch_size': {'values': [16, 32, 64]}
     }
}

# sweep_configの設定値でsweepを初期化する
sweep_id = wandb.sweep(sweep=sweep_config, project="kaggle-titanic")

Create sweep with ID: vp91c749
Sweep URL: https://wandb.ai/qhe202509-personal/kaggle-titanic/sweeps/vp91c749


In [46]:
# sweepを開始する
wandb.agent(sweep_id, function=train_model)

wandb: Agent Starting Run: y8yx3t7a with config:
wandb: 	batch_size: 16
wandb: 	hidden_dense_shape: 8
wandb: 	input_dense_shape: 8
wandb: 	optimizer: sgd


Epoch 1/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5762 - loss: 27.3514 - val_accuracy: 0.6306 - val_loss: 0.8800
Epoch 2/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6132 - loss: 0.7242 - val_accuracy: 0.6306 - val_loss: 0.7987
Epoch 3/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6164 - loss: 0.7036 - val_accuracy: 0.6269 - val_loss: 0.7627
Epoch 4/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 986us/step - accuracy: 0.6164 - loss: 0.6956 - val_accuracy: 0.6306 - val_loss: 0.7402
Epoch 5/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6148 - loss: 0.6885 - val_accuracy: 0.6306 - val_loss: 0.7214
Epoch 6/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 981us/step - accuracy: 0.6148 - loss: 0.6821 - val_accuracy: 0.6306 - val_loss: 0.7065
Epoch 7/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6148 - loss: 0.6779 - val_accuracy: 0.6306 - val_loss: 0.6928
Epoch 8/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 897us/step - accuracy: 0.6148 - loss: 0.6744 - val_accur

epoch/accuracy,▆▅▅▁▃▃▃▃▃▃▅▅▅▅▃▅▅▆▅▆▆███████████████████
epoch/epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,█████████▁▁▁▁▁▁▁██████████▁▁▁▁█▁▁▁▁▁▁█▁█
epoch/val_loss,█▄▄▃▄▄▄▅▆▆▇▇▄▃▃▃▃▁▁▁▂▁▂▂▁▃▄▄▃▃▄▃▄▃▅▃▇▆▇▆
epoch/accuracy,0.61798
epoch/epoch,282
epoch/learning_rate,0.01
epoch/loss,0.66259
epoch/val_accuracy,0.6306


wandb: Agent Starting Run: 0ksvc9w0 with config:
wandb: 	batch_size: 16
wandb: 	hidden_dense_shape: 8
wandb: 	input_dense_shape: 8
wandb: 	optimizer: rmsprop


Epoch 1/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6148 - loss: 32.7832 - val_accuracy: 0.5634 - val_loss: 15.3496
Epoch 2/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5297 - loss: 6.9876 - val_accuracy: 0.5224 - val_loss: 4.0817
Epoch 3/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 984us/step - accuracy: 0.4944 - loss: 3.3332 - val_accuracy: 0.5261 - val_loss: 2.9438
Epoch 4/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 939us/step - accuracy: 0.4992 - loss: 2.4492 - val_accuracy: 0.5187 - val_loss: 2.0265
Epoch 5/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step - accuracy: 0.4799 - loss: 1.6380 - val_accuracy: 0.4776 - val_loss: 1.4412
Epoch 6/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5201 - loss: 1.2400 - val_accuracy: 0.5299 - val_loss: 0.9880
Epoch 7/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 922us/step - accuracy: 0.5634 - loss: 0.9892 - val_accuracy: 0.6269 - val_loss: 0.8289
Epoch 8/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 904us/step - accuracy: 0.6148 - loss: 0.7956 - val_

...

epoch/accuracy,▁▇▆▇▆▇▇▆▇▇▆▇▇▇▇▇▇▇▆▆▇▇▇▇▇▇▆▇▇█▇▇▆▇▇▇▇█▇▇
epoch/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▃▂▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▂
epoch/val_accuracy,▁▅▃▅▆▅▆▅▆▁▃▅▆▃▃▆█▁▅▃▅▁▅▁▅▅▁▃▃▁▅▃▅▃▅▃▅▃▅▅
epoch/val_loss,█▆▂▂▅▂▄▂▁▆▅▅▂▂▂▂▃▂▄▅▃▄▃▅▃▆▃▄▄▃▅▅▃▃▃▃▃▃▃▃
epoch/accuracy,0.62119
epoch/epoch,118
epoch/learning_rate,0.01
epoch/loss,0.66545
epoch/val_accuracy,0.62687


wandb: Agent Starting Run: ya1evmeu with config:
wandb: 	batch_size: 16
wandb: 	hidden_dense_shape: 24
wandb: 	input_dense_shape: 8
wandb: 	optimizer: rmsprop


Epoch 1/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4960 - loss: 1.2148 - val_accuracy: 0.6455 - val_loss: 0.8277
Epoch 2/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5971 - loss: 0.7498 - val_accuracy: 0.6343 - val_loss: 0.9374
Epoch 3/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5987 - loss: 0.7324 - val_accuracy: 0.6381 - val_loss: 0.8589
Epoch 4/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6067 - loss: 0.7159 - val_accuracy: 0.6567 - val_loss: 0.7401
Epoch 5/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 995us/step - accuracy: 0.6212 - loss: 0.7284 - val_accuracy: 0.7090 - val_loss: 0.6772
Epoch 6/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 978us/step - accuracy: 0.6340 - loss: 0.6841 - val_accuracy: 0.7201 - val_loss: 0.6528
Epoch 7/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 988us/step - accuracy: 0.6501 - loss: 0.6838 - val_accuracy: 0.6978 - val_loss: 0.6416
Epoch 8/5000
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6485 - loss: 0.6668 - val_accura

epoch/accuracy,▁▁▂▁▂▃▃▄▄▅▆▇▇▇▇▇█▇████▇█▇████▇▇█▇█▇█████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇█████
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▇▇▆▅▅▅▅▄▃▃▃▃▃▃▂▂▂▃▂▂▂▁▂▁▁▂▁▂▂▁▁▁▁▂▁▂▁▁
epoch/val_accuracy,▃▄▂▃▄▂▃▅▅▅▆▆▁▅▆▅▂▇▅▅▇▇▆▄▇▇▅▇▅▅▇▆▅▇▇█▇▇▇▇
epoch/val_loss,▇▅▄▃█▃▃▃▇▄▂▂▂▂▁▂▁▂▃▂▁▂▃▁▁▁▁▅▂▁▁▁▂▁▁▁▄▁▁▃
epoch/accuracy,0.80417
epoch/epoch,238
epoch/learning_rate,0.001
epoch/loss,0.45719
epoch/val_accuracy,0.77612


wandb: Sweep Agent: Waiting for job.
wandb: Ctrl + C detected. Stopping sweep.


In [47]:
# グリッドサーチで得た最適なパラメータ値でモデルを作り直す
# （wandbは利用しない）
# テストデータによる予測も行なう
def train_model():
    # モデルの初期化とレイヤー定義
    model = tf.keras.Sequential([
        # 入力層
        tf.keras.Input(shape=(11,)),
        tf.keras.layers.Dense(16, activation='relu'),
        # 隠れ層
        tf.keras.layers.Dense(16, activation='relu'),
        # 出力層
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    # モデルの構築
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    # 学習の実施
    log = model.fit(X_train, Y_train,
                    epochs=5000,
                    batch_size=64,
                    verbose=True,
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(
                            monitor='val_loss',
                            min_delta=0,
                            patience=100,
                            verbose=1
                        )
                    ],
                    validation_data=(X_valid, Y_valid))

    # テストデータによる予測
    Y_pred_proba = model.predict(X_test)
    Y_pred = (Y_pred_proba > 0.5).astype("int32")

    # Y_predを返す
    return Y_pred

In [48]:
# train_modelを実行する
Y_pred = train_model()
Y_pred

Epoch 1/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6132 - loss: 26.9702 - val_accuracy: 0.6157 - val_loss: 8.6524
Epoch 2/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6035 - loss: 2.7771 - val_accuracy: 0.6828 - val_loss: 0.9899
Epoch 3/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6100 - loss: 1.0793 - val_accuracy: 0.6157 - val_loss: 2.6175
Epoch 4/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5795 - loss: 1.4083 - val_accuracy: 0.6418 - val_loss: 1.4326
Epoch 5/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6276 - loss: 1.1358 - val_accuracy: 0.6903 - val_loss: 0.9277
Epoch 6/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5730 - loss: 1.3508 - val_accuracy: 0.5597 - val_loss: 1.2702
Epoch 7/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6003 - loss: 1.1716 - val_accuracy: 0.6828 - val_loss: 1.1160
Epoch 8/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6116 - loss: 1.1657 - val_accuracy: 0

array([[0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
    

In [49]:
# X_testをDataFrameに戻し、X_test2に格納
X_test2 = pd.DataFrame(X_test, columns=test2.columns)

In [50]:
# Kaggleへ提出するためのデータが入ったDataFrameを作成
submission_data = pd.DataFrame()
submission_data["PassengerId"] = X_test2["PassengerId"].astype("int32")
submission_data["Survived"] = Y_pred
submission_data

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [51]:
# Kaggleに提出するためのCSVファイルを作成
submission_data.to_csv("my_submission_titanic.csv", index=False)